In [22]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from typing import TypedDict
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv

In [23]:
load_dotenv()

True

In [24]:
class State_(TypedDict):
    topic: str
    outline: str
    response: str

In [25]:

llm = HuggingFaceEndpoint(
    repo_id="google/gemma-4-31B-it",
    task="text-generation"
)
chat = ChatHuggingFace(llm=llm)

In [ ]:
def outline_generator (state: State_) -> State_:
    topic = state['topic']
    prompt = PromptTemplate(
        template="give me the outline of the given {topic}",
        input_variables=['topic']
    )
    response = prompt.invoke({'topic' : topic})

    result = chat.invoke(response)

    state['outline'] = result.content

    return state


In [32]:
def essay_generator (state: State_) -> State_:
    topic = state['topic']
    outline = state['outline']
    prompt = PromptTemplate(
            template="give me the eassy of 100 words on the given {topic} and the outline for the {topic} is {outline}",
            input_variables=['topic', 'outline']
        )
    fprompt = prompt.invoke({'topic': topic, 'outline':outline})

    result = chat.invoke(fprompt)
    state['response'] = result.content

    return state

In [28]:
# define state

graph = StateGraph(State_)

graph.add_node('outline_generator', outline_generator)
graph.add_node('essay_generator', essay_generator)

graph.add_edge(START, 'outline_generator')
graph.add_edge('outline_generator', 'essay_generator')
graph.add_edge('essay_generator', END)

workflow = graph.compile()

In [33]:
res = workflow.invoke({'topic': 'what is living a life means'})

res

{'topic': 'what is living a life means',
 'outline': AIMessage(content='Since you didn\'t provide a specific text, I have created a **comprehensive conceptual outline** based on the philosophical, psychological, and practical interpretations of the question: *"What does living a life mean?"*\n\nYou can use this outline to write an essay, a speech, or a journal entry.\n\n---\n\n# Outline: The Meaning of Living a Life\n\n## I. Introduction\n*   **The Hook:** A reflection on the difference between *existing* (biological survival) and *living* (experiencing life).\n*   **The Core Conflict:** The tension between inherited expectations (society/family) and personal authenticity.\n*   **Thesis Statement:** Living a life is not a destination or a single definition, but a continuous process of seeking purpose, cultivating connection, and embracing the full spectrum of human emotion.\n\n## II. The Biological vs. The Existential (Existing vs. Living)\n*   **Survival (Existing):** Meeting basic ne